In [2]:
import requests
import pandas as pd
import time

In [ ]:
# Load KAggle data to pandas dataframes
results = pd.read_csv("../data/results.csv")
races = pd.read_csv("../data/races.csv")
drivers = pd.read_csv("../data/drivers.csv")
constructors = pd.read_csv("../data/constructors.csv")

results shape: (26759, 18)
races shape: (1125, 18)
drivers shape: (861, 9)
constructors shape: (212, 5)


In [ ]:
# join everything together
df = results.merge(races[["raceId", "year", "round", "circuitId", "name"]], on="raceId")
df = df.merge(drivers[["driverId", "driverRef"]], on="driverId")
df = df.merge(constructors[["constructorId", "name"]], on="constructorId", suffixes=("_race", "_team"))

df = df[df["year"] >= 2010]

df = df[["year", "round", "name_race", "circuitId", "driverRef", "name_team", "grid", "position", "points", "statusId"]]

df.columns = ["season", "round", "race_name", "circuit", "driver", "team", "grid", "position", "points", "statusId"]

In [26]:
#Check data
print(f"Total rows: {len(df)}")
print(f"Seasons: {df['season'].min()} to {df['season'].max()}")
print(f"Unique drivers: {df['driver'].nunique()}")
print(f"Unique circuits: {df['circuit'].nunique()}")
print(df.head(10))

Total rows: 6436
Seasons: 2010 to 2024
Unique drivers: 80
Unique circuits: 35
       season  round           race_name  circuit              driver  \
20320    2010      1  Bahrain Grand Prix        3              alonso   
20321    2010      1  Bahrain Grand Prix        3               massa   
20322    2010      1  Bahrain Grand Prix        3            hamilton   
20323    2010      1  Bahrain Grand Prix        3              vettel   
20324    2010      1  Bahrain Grand Prix        3             rosberg   
20325    2010      1  Bahrain Grand Prix        3  michael_schumacher   
20326    2010      1  Bahrain Grand Prix        3              button   
20327    2010      1  Bahrain Grand Prix        3              webber   
20328    2010      1  Bahrain Grand Prix        3              liuzzi   
20329    2010      1  Bahrain Grand Prix        3         barrichello   

              team  grid position  points  statusId  
20320      Ferrari     3        1    25.0         1  
20321     

In [27]:
#Save cleaned data
df.to_csv("../data/f1_data.csv", index=False)
print("Saved!")

Saved!


In [ ]:
# Load circuits data and inspect available columns
circuits = pd.read_csv("../data/circuits.csv")
print(circuits.columns.tolist())

['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt', 'url']


In [ ]:
# Load status lookup table and check status codes
status = pd.read_csv("../data/status.csv")
print(status.head())

   statusId        status
0         1      Finished
1         2  Disqualified
2         3      Accident
3         4     Collision
4         5        Engine


In [ ]:
# Replace numeric circuit and status IDs with human-readable names
circuits_map = circuits[["circuitId", "circuitRef"]].set_index("circuitId")["circuitRef"].to_dict()
df["circuit"] = df["circuit"].map(circuits_map)

status_map = status[["statusId", "status"]].set_index("statusId")["status"].to_dict()
df["status"] = df["statusId"].map(status_map)

df = df.drop(columns=["statusId"])

print(df.head(10))

       season  round           race_name  circuit              driver  \
20320    2010      1  Bahrain Grand Prix  bahrain              alonso   
20321    2010      1  Bahrain Grand Prix  bahrain               massa   
20322    2010      1  Bahrain Grand Prix  bahrain            hamilton   
20323    2010      1  Bahrain Grand Prix  bahrain              vettel   
20324    2010      1  Bahrain Grand Prix  bahrain             rosberg   
20325    2010      1  Bahrain Grand Prix  bahrain  michael_schumacher   
20326    2010      1  Bahrain Grand Prix  bahrain              button   
20327    2010      1  Bahrain Grand Prix  bahrain              webber   
20328    2010      1  Bahrain Grand Prix  bahrain              liuzzi   
20329    2010      1  Bahrain Grand Prix  bahrain         barrichello   

              team  grid position  points    status  
20320      Ferrari     3        1    25.0  Finished  
20321      Ferrari     2        2    18.0  Finished  
20322      McLaren     4        3 

In [31]:
# save the clean final dataset
df.to_csv("../data/f1_data.csv", index=False)
print(f"Saved! {len(df)} rows")

Saved! 6436 rows


In [3]:
import requests
import time

all_rows = []

for round_num in range(1, 25):  # loop through up to 24 rounds
    url = f"https://api.jolpi.ca/ergast/f1/2025/{round_num}/results.json"
    response = requests.get(url)
    
    # stop if no data yet for this round
    if not response.text.strip():
        print(f"No data for round {round_num}, stopping")
        break
    
    try:
        data = response.json()
        races = data["MRData"]["RaceTable"]["Races"]
        if not races:
            print(f"No races found for round {round_num}, stopping")
            break
            
        race = races[0]
        for result in race["Results"]:
            all_rows.append({
                "season": 2025,
                "round": round_num,
                "race_name": race["raceName"],
                "circuit": race["Circuit"]["circuitId"],
                "driver": result["Driver"]["driverId"],
                "team": result["Constructor"]["name"],
                "grid": result["grid"],
                "position": result["position"],
                "points": result["points"],
                "status": result["status"]
            })
        
        print(f"Round {round_num} fetched — {race['raceName']}")
        time.sleep(1)
        
    except Exception as e:
        print(f"Error on round {round_num}: {e}")
        break

# combine with existing kaggle data
df_2025 = pd.DataFrame(all_rows)
print(f"\n2025 rows fetched: {len(df_2025)}")

Round 1 fetched — Australian Grand Prix
Round 2 fetched — Chinese Grand Prix
Round 3 fetched — Japanese Grand Prix
Round 4 fetched — Bahrain Grand Prix
Round 5 fetched — Saudi Arabian Grand Prix
Round 6 fetched — Miami Grand Prix
Round 7 fetched — Emilia Romagna Grand Prix
Round 8 fetched — Monaco Grand Prix
Round 9 fetched — Spanish Grand Prix
Round 10 fetched — Canadian Grand Prix
Round 11 fetched — Austrian Grand Prix
Round 12 fetched — British Grand Prix
Round 13 fetched — Belgian Grand Prix
Round 14 fetched — Hungarian Grand Prix
Round 15 fetched — Dutch Grand Prix
Round 16 fetched — Italian Grand Prix
Round 17 fetched — Azerbaijan Grand Prix
Round 18 fetched — Singapore Grand Prix
Round 19 fetched — United States Grand Prix
Round 20 fetched — Mexico City Grand Prix
Round 21 fetched — São Paulo Grand Prix
Round 22 fetched — Las Vegas Grand Prix
Round 23 fetched — Qatar Grand Prix
Round 24 fetched — Abu Dhabi Grand Prix

2025 rows fetched: 479


In [4]:
# load existing cleaned data
df_existing = pd.read_csv("../data/f1_data.csv")

# combine with 2025 data
df_full = pd.concat([df_existing, df_2025], ignore_index=True)

# save updated dataset
df_full.to_csv("../data/f1_data.csv", index=False)
print(f"Previous rows: {len(df_existing)}")
print(f"2025 rows added: {len(df_2025)}")
print(f"Total rows now: {len(df_full)}")

Previous rows: 6436
2025 rows added: 479
Total rows now: 6915
